## Augmentation with Step function and Linear interpolation with Jitter

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import src.pipeline as pipe
import src.visualization as visual

In [3]:
df = pipe.load_data()
display(df.head())

Dropped 9 unusable indicators.
Dataset loaded: 65 years (from 1960 to 2024), 27 variables.


,population_percent,population_growth,population_abs,employment_tot,employment_male,employment_female,forestarea_percent,forestarea_abs,agriland_percent,agriland_abs,...,fertilizer_percent,livestock_production_index,food_production_index,crop_production_index,cereal_production,cerealyield_abs,valueadded_percent,valueadded_dollars,exports_percent,imports_percent
1960-01-01,40.639,NaN,20400656.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-01-01,40.144,-0.557139,20287312.0,NaN,NaN,NaN,NaN,NaN,70.324028,206830.0,...,72.707716,70.91,85.93,93.68,13933400.0,2181.5,NaN,NaN,NaN,NaN
1962-01-01,39.645,-0.574190,20171158.0,NaN,NaN,NaN,NaN,NaN,70.218626,206520.0,...,70.071359,72.59,86.90,94.43,14433210.0,2225.3,NaN,NaN,2.563660,16.566057
1963-01-01,39.147,-0.534554,20063620.0,NaN,NaN,NaN,NaN,NaN,69.735813,205100.0,...,63.883735,65.95,86.37,97.19,13324660.0,2115.2,NaN,NaN,2.651714,14.571604
1964-01-01,38.650,-0.455075,19972523.0,NaN,NaN,NaN,NaN,NaN,69.572609,204620.0,...,64.593354,69.57,90.28,101.31,14007520.0,2243.0,NaN,NaN,2.792923,15.115329


In [4]:
# dizionario per salvare in memoria i subset originali e i train aumentati con step e linear
orig_aug_subsets = {}

for col in df.columns:
    print('='*30, col.upper(), '='*30)
    subset = df[[col]].copy()
    subset['Year'] = subset.index.year
    subset.rename(columns={col: 'Value'}, inplace=True)
    subset = subset[['Year', 'Value']]
    subset = subset.reset_index(drop=True)
    subset = subset.dropna()
    train_orig, val_orig, test_orig = pipe.split_train_val_test(subset)
    
    # faccio augmentation SOLO sul train set: aumento sia anni che valori
    x_train_vals = train_orig['Year']
    y_train_vals = train_orig['Value']
    
    df_step = pipe.augment_step_function(x_train_vals, y_train_vals, scale_factor=10)
    df_jitter = pipe.augment_linear_with_jitter(x_train_vals, y_train_vals, scale_factor=10, noise_level=0.05)
    visual.plot_augmented(col, df_step, df_jitter, x_train_vals, y_train_vals)
    orig_aug_subsets[col] = {
        'full_orig': subset,
        'orig_train': train_orig,
        'step_aug_train': df_step,
        'jitter_aug_train': df_jitter,
        'orig_val': val_orig,
        'orig_test': test_orig
    }

============================== POPULATION_PERCENT ==============================
Train: 1960-2004 (45 obs)
Val:   2005-2014 (10 obs)
Test:  2015-2024 (10 obs)
Grafico salvato in: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\07_AUGMENTATION\AUG_population_percent.png
============================== POPULATION_GROWTH ==============================
Train: 1961-2004 (44 obs)
Val:   2005-2015 (11 obs)
Test:  2016-2024 (9 obs)
Grafico salvato in: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\07_AUGMENTATION\AUG_population_growth.png
============================== POPULATION_ABS ==============================
Train: 1960-2004 (45 obs)
Val:   2005-2014 (10 obs)
Test:  2015-2024 (10 obs)
Grafico salvato in: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\07_AUGMENTATION\AUG_population_abs.png
============================== EMPLOYMENT_TOT ==============================
Train: 1991-2013 (23 obs)
Val:   2014-2018 (5 obs